In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Import Basic Libraries
# ═══════════════════════════════════════════════════════════
import json
import pandas as pd
import numpy as np
import glob
import os
import warnings
from collections import defaultdict
warnings.filterwarnings('ignore')

print("✅ Basic imports done")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Import Deep Learning
# ═══════════════════════════════════════════════════════════
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"✅ TensorFlow {tf.__version__}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: Import ML & Visualization
# ═══════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
print("✅ All tools ready")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: Create Project Structure
# ═══════════════════════════════════════════════════════════
folders = ['data', 'models', 'plots', 'analysis']
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: Load Data Files
# ═══════════════════════════════════════════════════════════
JSON_PATH = r"C:\Users\VASU MONPARA\OneDrive\Desktop\Cricket\Dataset\ipl_json"
files = glob.glob(JSON_PATH + "/*.json")

print(f"📂 Found {len(files)} match files")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: Extract Complete Match Data
# ═══════════════════════════════════════════════════════════
print("🔄 Extracting match data...")

matches = []

for f in tqdm(files, desc="Matches"):
    try:
        with open(f, 'r', encoding='utf-8') as file:
            data = json.load(file)
        
        info = data['info']
        outcome = info.get('outcome', {})
        toss = info.get('toss', {})
        
        teams = info.get('teams', [])
        if len(teams) != 2:
            continue
        
        winner = outcome.get('winner')
        if not winner:
            continue
        
        matches.append({
            'match_id': os.path.basename(f).replace('.json', ''),
            'season': info.get('season'),
            'city': info.get('city', 'Unknown'),
            'venue': info.get('venue', 'Unknown'),
            'match_type': info.get('match_type', 'T20'),
            'team1': teams[0],
            'team2': teams[1],
            'toss_winner': toss.get('winner', teams[0]),
            'toss_decision': toss.get('decision', 'bat'),
            'winner': winner,
            'win_by_runs': outcome.get('by', {}).get('runs', 0),
            'win_by_wickets': outcome.get('by', {}).get('wickets', 0),
            'player_of_match': info.get('player_of_match', [None])[0]
        })
    except:
        continue

matches_df = pd.DataFrame(matches)
print(f"✅ Loaded {len(matches_df)} matches")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: Extract Ball-by-Ball Data with ALL Details
# ═══════════════════════════════════════════════════════════
print("🔄 Extracting ball-by-ball data...")

balls = []

for f in tqdm(files, desc="Ball data"):
    try:
        with open(f, 'r', encoding='utf-8') as file:
            data = json.load(file)
        
        match_id = os.path.basename(f).replace('.json', '')
        info = data['info']
        teams = info.get('teams', [])
        venue = info.get('venue', 'Unknown')
        city = info.get('city', 'Unknown')
        winner = info['outcome'].get('winner')
        
        if not winner or len(teams) != 2:
            continue
        
        for inning_num, inning in enumerate(data['innings']):
            batting_team = inning['team']
            bowling_team = teams[1] if batting_team == teams[0] else teams[0]
            target = inning.get('target', {}).get('runs', 0)
            
            runs = 0
            wickets = 0
            ball_count = 0
            
            for over in inning['overs']:
                over_num = over['over']
                
                for delivery in over['deliveries']:
                    ball_count += 1
                    
                    # Get delivery details
                    batsman = delivery['batter']
                    bowler = delivery['bowler']
                    non_striker = delivery.get('non_striker', 'Unknown')
                    
                    # Runs
                    runs_data = delivery['runs']
                    runs_off_bat = runs_data['batter']
                    extras = runs_data['extras']
                    total_runs = runs_data['total']
                    runs += total_runs
                    
                    # Wicket
                    is_wicket = 1 if 'wickets' in delivery else 0
                    wickets += is_wicket
                    
                    # Extras breakdown
                    extras_info = delivery.get('extras', {})
                    wide = extras_info.get('wides', 0)
                    noball = extras_info.get('noballs', 0)
                    bye = extras_info.get('byes', 0)
                    legbye = extras_info.get('legbyes', 0)
                    
                    # Legal ball
                    legal_ball = 1 if (wide == 0 and noball == 0) else 0
                    
                    # Calculate rates
                    balls_faced = ball_count
                    current_rr = (runs / balls_faced * 6) if balls_faced > 0 else 0
                    
                    if target > 0:
                        runs_needed = max(0, target - runs)
                        balls_left = 120 - balls_faced
                        req_rr = (runs_needed / balls_left * 6) if balls_left > 0 else 0
                    else:
                        runs_needed = 0
                        req_rr = 0
                    
                    # In powerplay?
                    in_powerplay = 1 if over_num < 6 else 0
                    
                    balls.append({
                        'match_id': match_id,
                        'venue': venue,
                        'city': city,
                        'inning': inning_num + 1,
                        'batting_team': batting_team,
                        'bowling_team': bowling_team,
                        'over': over_num,
                        'ball': ball_count,
                        'batsman': batsman,
                        'bowler': bowler,
                        'non_striker': non_striker,
                        'runs_off_bat': runs_off_bat,
                        'extras': extras,
                        'wide': wide,
                        'noball': noball,
                        'bye': bye,
                        'legbye': legbye,
                        'legal_ball': legal_ball,
                        'is_wicket': is_wicket,
                        'total_runs': total_runs,
                        'score': runs,
                        'wickets': wickets,
                        'current_rr': round(current_rr, 2),
                        'target': target,
                        'runs_needed': runs_needed,
                        'required_rr': round(req_rr, 2),
                        'in_powerplay': in_powerplay,
                        'winner': winner
                    })
    except:
        continue

balls_df = pd.DataFrame(balls)
print(f"✅ Loaded {len(balls_df):,} balls")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8: Calculate Team Statistics
# ═══════════════════════════════════════════════════════════
print("📊 Calculating team stats...")

all_teams = sorted(set(matches_df['team1']) | set(matches_df['team2']))

team_stats = {}
for team in all_teams:
    played = len(matches_df[(matches_df['team1'] == team) | (matches_df['team2'] == team)])
    won = len(matches_df[matches_df['winner'] == team])
    
    team_stats[team] = {
        'matches': played,
        'wins': won,
        'losses': played - won,
        'win_rate': won / played if played > 0 else 0
    }

print(f"✅ Stats for {len(team_stats)} teams")



In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9: Calculate Venue Statistics
# ═══════════════════════════════════════════════════════════
print("📊 Calculating venue stats...")

venue_stats = {}
for venue in matches_df['venue'].unique():
    venue_matches = matches_df[matches_df['venue'] == venue]
    
    # First innings scores
    venue_balls = balls_df[(balls_df['venue'] == venue) & (balls_df['inning'] == 1)]
    if len(venue_balls) > 0:
        avg_score = venue_balls.groupby('match_id')['score'].max().mean()
    else:
        avg_score = 160
    
    # Toss decisions
    bat_first = len(venue_matches[venue_matches['toss_decision'] == 'bat'])
    field_first = len(venue_matches[venue_matches['toss_decision'] == 'field'])
    
    venue_stats[venue] = {
        'matches': len(venue_matches),
        'avg_score': round(avg_score, 1),
        'bat_first_count': bat_first,
        'field_first_count': field_first,
        'prefer_bat': bat_first > field_first
    }

print(f"✅ Stats for {len(venue_stats)} venues")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10: Calculate Player Batting Statistics
# ═══════════════════════════════════════════════════════════
print("📊 Calculating player stats...")

# Batting stats
batsman_stats = balls_df[balls_df['legal_ball'] == 1].groupby('batsman').agg({
    'runs_off_bat': 'sum',
    'ball': 'count',
    'is_wicket': 'sum',
    'match_id': 'nunique'
}).reset_index()

batsman_stats.columns = ['player', 'runs', 'balls', 'dismissals', 'matches']
batsman_stats['avg'] = batsman_stats.apply(
    lambda x: round(x['runs'] / x['dismissals'], 2) if x['dismissals'] > 0 else x['runs'], 
    axis=1
)
batsman_stats['sr'] = batsman_stats.apply(
    lambda x: round(x['runs'] / x['balls'] * 100, 2) if x['balls'] > 0 else 0, 
    axis=1
)

print(f"✅ Stats for {len(batsman_stats)} batsmen")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11: Calculate Player Bowling Statistics
# ═══════════════════════════════════════════════════════════
bowler_stats = balls_df[balls_df['legal_ball'] == 1].groupby('bowler').agg({
    'total_runs': 'sum',
    'ball': 'count',
    'is_wicket': 'sum',
    'match_id': 'nunique'
}).reset_index()

bowler_stats.columns = ['player', 'runs_given', 'balls', 'wickets', 'matches']
bowler_stats['economy'] = bowler_stats.apply(
    lambda x: round(x['runs_given'] / x['balls'] * 6, 2) if x['balls'] > 0 else 0, 
    axis=1
)
bowler_stats['avg'] = bowler_stats.apply(
    lambda x: round(x['runs_given'] / x['wickets'], 2) if x['wickets'] > 0 else 999, 
    axis=1
)

print(f"✅ Stats for {len(bowler_stats)} bowlers")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12: Head-to-Head Team Analysis
# ═══════════════════════════════════════════════════════════
print("📊 Calculating head-to-head stats...")

h2h_stats = {}

for team1 in all_teams:
    for team2 in all_teams:
        if team1 >= team2:
            continue
        
        h2h_matches = matches_df[
            ((matches_df['team1'] == team1) & (matches_df['team2'] == team2)) |
            ((matches_df['team1'] == team2) & (matches_df['team2'] == team1))
        ]
        
        team1_wins = len(h2h_matches[h2h_matches['winner'] == team1])
        team2_wins = len(h2h_matches[h2h_matches['winner'] == team2])
        
        key = f"{team1} vs {team2}"
        h2h_stats[key] = {
            'matches': len(h2h_matches),
            f'{team1}_wins': team1_wins,
            f'{team2}_wins': team2_wins
        }

print(f"✅ Calculated {len(h2h_stats)} head-to-head records")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13: Player vs Bowler Analysis
# ═══════════════════════════════════════════════════════════
print("📊 Calculating batsman vs bowler stats...")

bat_vs_bowl = balls_df[balls_df['legal_ball'] == 1].groupby(['batsman', 'bowler']).agg({
    'runs_off_bat': 'sum',
    'ball': 'count',
    'is_wicket': 'sum'
}).reset_index()

bat_vs_bowl.columns = ['batsman', 'bowler', 'runs', 'balls', 'dismissals']
bat_vs_bowl['sr'] = (bat_vs_bowl['runs'] / bat_vs_bowl['balls'] * 100).round(2)

# Filter significant matchups (at least 10 balls)
bat_vs_bowl = bat_vs_bowl[bat_vs_bowl['balls'] >= 10]

print(f"✅ Found {len(bat_vs_bowl)} significant matchups")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14: Create Encodings
# ═══════════════════════════════════════════════════════════
team_to_id = {team: i for i, team in enumerate(sorted(all_teams))}
id_to_team = {i: team for team, i in team_to_id.items()}

venue_to_id = {v: i for i, v in enumerate(sorted(matches_df['venue'].unique()))}
city_to_id = {c: i for i, c in enumerate(sorted(matches_df['city'].unique()))}

player_to_id = {p: i for i, p in enumerate(sorted(set(balls_df['batsman']) | set(balls_df['bowler'])))}

print(f"✅ Encoded {len(team_to_id)} teams, {len(venue_to_id)} venues, {len(player_to_id)} players")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 15: Prepare Match Winner Features
# ═══════════════════════════════════════════════════════════
print("🔧 Creating match winner features...")

match_features = matches_df.copy()

# Encode teams
match_features['team1_id'] = match_features['team1'].map(team_to_id)
match_features['team2_id'] = match_features['team2'].map(team_to_id)
match_features['venue_id'] = match_features['venue'].map(venue_to_id)
match_features['city_id'] = match_features['city'].map(city_to_id)

# Toss features
match_features['toss_winner_id'] = match_features['toss_winner'].map(team_to_id)
match_features['toss_bat'] = (match_features['toss_decision'] == 'bat').astype(int)
match_features['team1_won_toss'] = (match_features['team1'] == match_features['toss_winner']).astype(int)

# Team form
match_features['team1_win_rate'] = match_features['team1'].map(lambda x: team_stats[x]['win_rate'])
match_features['team2_win_rate'] = match_features['team2'].map(lambda x: team_stats[x]['win_rate'])

# Venue advantage
match_features['venue_avg_score'] = match_features['venue'].map(
    lambda x: venue_stats[x]['avg_score'] if x in venue_stats else 160
)

# Target
match_features['team1_won'] = (match_features['team1'] == match_features['winner']).astype(int)

# Clean
match_features = match_features.dropna()

print(f"✅ Created features for {len(match_features)} matches")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 16: Build Match Winner Model (LSTM)
# ═══════════════════════════════════════════════════════════
print("🏗️ Building Match Winner LSTM...")

feature_cols = ['team1_id', 'team2_id', 'venue_id', 'city_id', 'toss_winner_id',
                'toss_bat', 'team1_won_toss', 'team1_win_rate', 'team2_win_rate', 
                'venue_avg_score']

X_match = match_features[feature_cols].values
y_match = match_features['team1_won'].values

# Reshape for LSTM (add time dimension)
X_match = X_match.reshape(X_match.shape[0], 1, X_match.shape[1])

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_match, y_match, test_size=0.2, random_state=42
)

# Build model
match_model = keras.Sequential([
    layers.LSTM(64, input_shape=(1, len(feature_cols))),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='MatchWinnerLSTM')

match_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("✅ Match winner model built")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 17: Train Match Winner Model
# ═══════════════════════════════════════════════════════════
print("🔄 Training match winner model...")

hist_match = match_model.fit(
    X_train_m, y_train_m,
    validation_data=(X_test_m, y_test_m),
    epochs=50,
    batch_size=32,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ],
    verbose=0
)

score_match = match_model.evaluate(X_test_m, y_test_m, verbose=0)
print(f"✅ Match Winner Accuracy: {score_match[1]*100:.2f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 18: Prepare Ball-by-Ball Win Probability Features
# ═══════════════════════════════════════════════════════════
print("🔧 Creating ball-by-ball features...")

# Filter 2nd innings
chase_data = balls_df[balls_df['inning'] == 2].copy()
chase_data = chase_data[chase_data['target'] > 0]

# Features
chase_data['batting_team_id'] = chase_data['batting_team'].map(team_to_id)
chase_data['bowling_team_id'] = chase_data['bowling_team'].map(team_to_id)
chase_data['batsman_id'] = chase_data['batsman'].map(player_to_id)
chase_data['bowler_id'] = chase_data['bowler'].map(player_to_id)
chase_data['venue_id'] = chase_data['venue'].map(venue_to_id)

chase_data['balls_left'] = 120 - chase_data['ball']
chase_data['wickets_left'] = 10 - chase_data['wickets']
chase_data['won'] = (chase_data['batting_team'] == chase_data['winner']).astype(int)

chase_data = chase_data.dropna()

print(f"✅ Created {len(chase_data):,} ball-by-ball records")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 19: Create Sequences for Ball-by-Ball Model
# ═══════════════════════════════════════════════════════════
print("🔧 Creating sequences...")

seq_length = 30  # Last 30 balls

ball_features = ['runs_off_bat', 'is_wicket', 'score', 'wickets', 'current_rr', 
                 'required_rr', 'in_powerplay', 'wide', 'noball']

sequences = []
targets = []

for match_id in chase_data['match_id'].unique():
    match_balls = chase_data[chase_data['match_id'] == match_id]
    
    if len(match_balls) < seq_length:
        continue
    
    for i in range(len(match_balls) - seq_length):
        seq = match_balls.iloc[i:i+seq_length][ball_features].values
        target = match_balls.iloc[i+seq_length-1]['won']
        
        sequences.append(seq)
        targets.append(target)

X_ball = np.array(sequences)
y_ball = np.array(targets)

print(f"✅ Created {len(X_ball):,} sequences")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 20: Build Ball-by-Ball Models (Multiple)
# ═══════════════════════════════════════════════════════════
print("🏗️ Building ball-by-ball models...")

# Model 1: Simple LSTM
model_ball_1 = keras.Sequential([
    layers.LSTM(64, input_shape=(seq_length, len(ball_features))),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='SimpleLSTM')

# Model 2: Stacked LSTM
model_ball_2 = keras.Sequential([
    layers.LSTM(128, return_sequences=True, input_shape=(seq_length, len(ball_features))),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='StackedLSTM')

# Model 3: Bidirectional LSTM
model_ball_3 = keras.Sequential([
    layers.Bidirectional(layers.LSTM(128, return_sequences=True), 
                         input_shape=(seq_length, len(ball_features))),
    layers.Dropout(0.3),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='BiLSTM')

# Model 4: GRU
model_ball_4 = keras.Sequential([
    layers.GRU(128, return_sequences=True, input_shape=(seq_length, len(ball_features))),
    layers.Dropout(0.3),
    layers.GRU(64),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='GRU')

for model in [model_ball_1, model_ball_2, model_ball_3, model_ball_4]:
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("✅ Built 4 ball-by-ball models")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 21: Train All Ball-by-Ball Models
# ═══════════════════════════════════════════════════════════
print("🔄 Training all models...\n")

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_ball, y_ball, test_size=0.2, random_state=42
)

models_ball = [model_ball_1, model_ball_2, model_ball_3, model_ball_4]
histories_ball = []
scores_ball = []

for i, model in enumerate(models_ball, 1):
    print(f"Training {model.name}...")
    
    hist = model.fit(
        X_train_b, y_train_b,
        validation_data=(X_test_b, y_test_b),
        epochs=15,
        batch_size=64,
        callbacks=[ 
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
        ],
        verbose=0
    )
    
    score =                                   model.evaluate(X_test_b, y_test_b, verbose=0)
    
    histories_ball.append(hist)
    scores_ball.append(score)
    
    print(f"  Accuracy: {score[1]*100:.2f}%\n")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 22: Select Best Ball-by-Ball Model
# ═══════════════════════════════════════════════════════════
best_idx = np.argmax([s[1] for s in scores_ball])
best_ball_model = models_ball[best_idx]
best_ball_score = scores_ball[best_idx][1]

print(f"🏆 Best Model: {best_ball_model.name}")
print(f"🎯 Accuracy: {best_ball_score*100:.2f}%")


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 23: Visualization - Team Performance
# ═══════════════════════════════════════════════════════════
print("\n📊 Creating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Top teams by wins
team_wins = matches_df['winner'].value_counts().head(10)
axes[0, 0].barh(team_wins.index, team_wins.values, color=sns.color_palette("viridis", 10))
axes[0, 0].set_xlabel('Wins')
axes[0, 0].set_title('🏆 Top 10 Teams by Wins', fontweight='bold')

# Toss decisions
toss_counts = matches_df['toss_decision'].value_counts()
axes[0, 1].pie(toss_counts.values, labels=toss_counts.index, autopct='%1.1f%%', 
               colors=['#FF6B6B', '#4ECDC4'])
axes[0, 1].set_title('🎯 Toss Decisions', fontweight='bold')

# Venue match counts
venue_counts = matches_df['venue'].value_counts().head(10)
axes[1, 0].barh(venue_counts.index, venue_counts.values, color=sns.color_palette("coolwarm", 10))
axes[1, 0].set_xlabel('Matches')
axes[1, 0].set_title('📍 Top Venues', fontweight='bold')

# Runs distribution
runs_dist = balls_df['runs_off_bat'].value_counts().sort_index()
axes[1, 1].bar(runs_dist.index, runs_dist.values, color=sns.color_palette("rocket", len(runs_dist)))
axes[1, 1].set_xlabel('Runs')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('📈 Runs Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/team_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Saved: team_analysis.png")
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 24: Visualization - Model Performance
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model comparison
model_names = [m.name for m in models_ball]
model_accs = [s[1] * 100 for s in scores_ball]

colors = sns.color_palette("husl", len(models_ball))
bars = axes[0].barh(model_names, model_accs, color=colors)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('🏆 Model Comparison', fontweight='bold', fontsize=14)
axes[0].set_xlim(0, 100)

for bar in bars:
    width = bar.get_width()
    axes[0].text(width + 1, bar.get_y() + bar.get_height()/2, 
                f'{width:.1f}%', va='center')

# Training history of best model
best_hist = histories_ball[best_idx]
axes[1].plot(best_hist.history['accuracy'], label='Training', linewidth=2)
axes[1].plot(best_hist.history['val_accuracy'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'📈 {best_ball_model.name} Training', fontweight='bold', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plots/model_performance.png', dpi=300, bbox_inches='tight')
print("✅ Saved: model_performance.png")
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 25: Visualization - Player Performance
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Top batsmen
top_batsmen = batsman_stats.nlargest(10, 'runs')
axes[0, 0].barh(top_batsmen['player'], top_batsmen['runs'], 
                color=sns.color_palette("plasma", 10))
axes[0, 0].set_xlabel('Runs')
axes[0, 0].set_title('🏏 Top 10 Run Scorers', fontweight='bold')

# Best strike rates (min 500 balls)
top_sr = batsman_stats[batsman_stats['balls'] >= 500].nlargest(10, 'sr')
axes[0, 1].barh(top_sr['player'], top_sr['sr'], 
                color=sns.color_palette("viridis", 10))
axes[0, 1].set_xlabel('Strike Rate')
axes[0, 1].set_title('⚡ Best Strike Rates', fontweight='bold')

# Top wicket takers
top_bowlers = bowler_stats.nlargest(10, 'wickets')
axes[1, 0].barh(top_bowlers['player'], top_bowlers['wickets'], 
                color=sns.color_palette("coolwarm", 10))
axes[1, 0].set_xlabel('Wickets')
axes[1, 0].set_title('🎯 Top Wicket Takers', fontweight='bold')

# Best economy (min 100 balls)
top_eco = bowler_stats[bowler_stats['balls'] >= 100].nsmallest(10, 'economy')
axes[1, 1].barh(top_eco['player'], top_eco['economy'], 
                color=sns.color_palette("rocket_r", 10))
axes[1, 1].set_xlabel('Economy')
axes[1, 1].set_title('💰 Best Economy Rates', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/player_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Saved: player_analysis.png")
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 26: Confusion Matrix
# ═══════════════════════════════════════════════════════════
predictions = best_ball_model.predict(X_test_b, verbose=0)
pred_classes = (predictions > 0.5).astype(int).flatten()

cm = confusion_matrix(y_test_b, pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'🎯 Confusion Matrix - {best_ball_model.name}', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plots/confusion_matrix.png', dpi=300, bbox_inches='tight')
print("✅ Saved: confusion_matrix.png")
plt.show()

In [ ]:
import pickle

print("\n💾 Saving models and data (optimized)...")

# ─────────────────────────────────────────────
# Save main LSTM models (Keras format)
# ─────────────────────────────────────────────
match_model.save("models/match_winner.keras")
best_ball_model.save("models/ball_by_ball.keras")

# ─────────────────────────────────────────────
# Save all ball models
# ─────────────────────────────────────────────
for i, model in enumerate(models_ball, 1):
    model.save(f"models/ball_models/ball_model_{i}.keras")

print("✅ Neural models saved")


save_data = {
    "team_to_id": team_to_id,
    "id_to_team": id_to_team,
    "venue_to_id": venue_to_id,
    "city_to_id": city_to_id,
    "player_to_id": player_to_id,
    "team_stats": team_stats,
    "venue_stats": venue_stats,
    "batsman_stats": batsman_stats.to_dict("records"),
    "bowler_stats": bowler_stats.to_dict("records"),
    "h2h_stats": h2h_stats,
    "feature_cols": feature_cols,
    "ball_features": ball_features
}

with open("models/cricket_data.pkl", "wb") as f:
    pickle.dump(save_data, f, protocol=pickle.HIGHEST_PROTOCOL)

print("✅ Data encodings saved")



In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 28: Create Analysis Reports
# ═══════════════════════════════════════════════════════════
print("\n📝 Creating analysis reports...")

# Save team stats
team_stats_df = pd.DataFrame(team_stats).T
team_stats_df.to_csv('analysis/team_statistics.csv')

# Save venue stats
venue_stats_df = pd.DataFrame(venue_stats).T
venue_stats_df.to_csv('analysis/venue_statistics.csv')

# Save player stats
batsman_stats.to_csv('analysis/batsman_statistics.csv', index=False)
bowler_stats.to_csv('analysis/bowler_statistics.csv', index=False)

# Save head to head
h2h_df = pd.DataFrame(h2h_stats).T
h2h_df.to_csv('analysis/head_to_head.csv')

# Save batsman vs bowler
bat_vs_bowl.to_csv('analysis/batsman_vs_bowler.csv', index=False)

print("✅ All analysis reports saved")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 29: Match Prediction Function
# ═══════════════════════════════════════════════════════════
def predict_match_winner(team1, team2, venue, toss_winner, toss_decision):
    """
    Predict match winner
    
    Args:
        team1: First team name
        team2: Second team name
        venue: Venue name
        toss_winner: Team that won toss
        toss_decision: 'bat' or 'field'
    
    Returns:
        Dictionary with predictions
    """
    try:
        # Encode inputs
        team1_id = team_to_id.get(team1, 0)
        team2_id = team_to_id.get(team2, 0)
        venue_id = venue_to_id.get(venue, 0)
        city = 'Unknown'
        city_id = city_to_id.get(city, 0)
        toss_winner_id = team_to_id.get(toss_winner, 0)
        toss_bat = 1 if toss_decision == 'bat' else 0
        team1_won_toss = 1 if toss_winner == team1 else 0
        
        team1_wr = team_stats.get(team1, {}).get('win_rate', 0.5)
        team2_wr = team_stats.get(team2, {}).get('win_rate', 0.5)
        venue_avg = venue_stats.get(venue, {}).get('avg_score', 160)
        
        # Create feature vector
        features = np.array([[team1_id, team2_id, venue_id, city_id, toss_winner_id,
                             toss_bat, team1_won_toss, team1_wr, team2_wr, venue_avg]])
        features = features.reshape(1, 1, -1)
        
        # Predict
        prob = match_model.predict(features, verbose=0)[0][0]
        
        return {
            'team1': team1,
            'team2': team2,
            'team1_win_prob': round(prob * 100, 2),
            'team2_win_prob': round((1 - prob) * 100, 2),
            'predicted_winner': team1 if prob > 0.5 else team2
        }
    except Exception as e:
        return {'error': str(e)}

print("✅ Match prediction function ready")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 30: Ball-by-Ball Win Probability Function
# ═══════════════════════════════════════════════════════════
def calculate_win_probability(sequence_data):
    """
    Calculate win probability from ball sequence
    
    Args:
        sequence_data: Array of shape (30, 9) with last 30 balls
    
    Returns:
        Win probability percentage
    """
    try:
        seq = np.array([sequence_data])
        prob = best_ball_model.predict(seq, verbose=0)[0][0]
        return round(prob * 100, 2)
    except Exception as e:
        return None

print("✅ Win probability function ready")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 31: Player Performance Prediction Function
# ═══════════════════════════════════════════════════════════
def predict_player_performance(player_name, opponent_team, venue):
    """
    Predict player performance in upcoming match
    
    Args:
        player_name: Player name
        opponent_team: Opponent team name
        venue: Venue name
    
    Returns:
        Dictionary with predictions
    """
    # Get player stats
    bat_stats = batsman_stats[batsman_stats['player'] == player_name]
    bowl_stats = bowler_stats[bowler_stats['player'] == player_name]
    
    result = {'player': player_name}
    
    # Batting prediction
    if not bat_stats.empty:
        stats = bat_stats.iloc[0]
        result['batting'] = {
            'avg': stats['avg'],
            'strike_rate': stats['sr'],
            'total_runs': stats['runs'],
            'predicted_runs': int(stats['avg'] * 0.8)  # Conservative estimate
        }
    
    # Bowling prediction
    if not bowl_stats.empty:
        stats = bowl_stats.iloc[0]
        result['bowling'] = {
            'economy': stats['economy'],
            'avg': stats['avg'],
            'total_wickets': stats['wickets'],
            'predicted_wickets': 1 if stats['wickets'] / stats['matches'] > 1 else 0
        }
    
    return result

print("✅ Player prediction function ready")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 32: Test All Prediction Functions
# ═══════════════════════════════════════════════════════════
print("\n🧪 Testing prediction functions...\n")

# Test match prediction
if len(all_teams) >= 2:
    test_teams = list(all_teams)[:2]
    test_venue = list(venue_to_id.keys())[0]
    
    match_pred = predict_match_winner(
        team1=test_teams[0],
        team2=test_teams[1],
        venue=test_venue,
        toss_winner=test_teams[0],
        toss_decision='bat'
    )
    
    print("📊 Match Prediction Test:")
    print(f"   {match_pred['team1']} vs {match_pred['team2']}")
    print(f"   {match_pred['team1']}: {match_pred['team1_win_prob']}%")
    print(f"   {match_pred['team2']}: {match_pred['team2_win_prob']}%")
    print(f"   Predicted Winner: {match_pred['predicted_winner']}")

# Test ball-by-ball
test_sequence = np.random.rand(30, 9)
win_prob = calculate_win_probability(test_sequence)
print(f"\n📈 Ball-by-Ball Test:")
print(f"   Win Probability: {win_prob}%")

# Test player prediction
if not batsman_stats.empty:
    test_player = batsman_stats.iloc[0]['player']
    player_pred = predict_player_performance(test_player, test_teams[1], test_venue)
    print(f"\n🏏 Player Prediction Test:")
    print(f"   Player: {player_pred['player']}")
    if 'batting' in player_pred:
        print(f"   Predicted Runs: {player_pred['batting']['predicted_runs']}")



In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 33: Generate Final Summary Report
# ═══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("📊 FINAL SUMMARY REPORT")
print("="*70)

print(f"\n📁 Dataset Overview:")
print(f"   Total Matches: {len(matches_df):,}")
print(f"   Total Balls: {len(balls_df):,}")
print(f"   Teams: {len(all_teams)}")
print(f"   Venues: {len(venue_to_id)}")
print(f"   Players: {len(player_to_id)}")

print(f"\n🏆 Models Trained:")
print(f"   1. Match Winner LSTM")
print(f"   2. Ball-by-Ball Simple LSTM")
print(f"   3. Ball-by-Ball Stacked LSTM")
print(f"   4. Ball-by-Ball Bidirectional LSTM")
print(f"   5. Ball-by-Ball GRU")

print(f"\n🎯 Model Performance:")
print(f"   Match Winner Accuracy: {score_match[1]*100:.2f}%")
print(f"   Best Ball-by-Ball Model: {best_ball_model.name}")
print(f"   Ball-by-Ball Accuracy: {best_ball_score*100:.2f}%")

print(f"\n📊 Statistical Analysis:")
print(f"   Team Statistics: {len(team_stats)} teams analyzed")
print(f"   Venue Statistics: {len(venue_stats)} venues analyzed")
print(f"   Batsman Statistics: {len(batsman_stats)} players")
print(f"   Bowler Statistics: {len(bowler_stats)} players")
print(f"   Head-to-Head Records: {len(h2h_stats)} matchups")
print(f"   Batsman vs Bowler: {len(bat_vs_bowl)} significant matchups")

print(f"\n💾 Files Saved:")
print(f"   Models:")
print(f"     • match_winner_lstm.h5")
print(f"     • ball_by_ball_lstm.h5")
print(f"     • ball_model_1_SimpleLSTM.h5")
print(f"     • ball_model_2_StackedLSTM.h5")
print(f"     • ball_model_3_BiLSTM.h5")
print(f"     • ball_model_4_GRU.h5")
print(f"     • cricket_data.pkl")
print(f"   \n   Analysis Reports:")
print(f"     • team_statistics.csv")
print(f"     • venue_statistics.csv")
print(f"     • batsman_statistics.csv")
print(f"     • bowler_statistics.csv")
print(f"     • head_to_head.csv")
print(f"     • batsman_vs_bowler.csv")
print(f"   \n   Visualizations:")
print(f"     • team_analysis.png")
print(f"     • model_performance.png")
print(f"     • player_analysis.png")
print(f"     • confusion_matrix.png")

print(f"\n🎮 Prediction Features:")
print(f"   ✅ Match Winner Prediction")
print(f"   ✅ Ball-by-Ball Win Probability")
print(f"   ✅ Player Performance Prediction")
print(f"   ✅ Head-to-Head Analysis")
print(f"   ✅ Venue Analysis")
print(f"   ✅ Team Form Analysis")

print("\n" + "="*70)
print("✅ PROJECT COMPLETE!")
print("="*70)
print("\n🚀 You can now:")
print("   1. Predict match winners before the match")
print("   2. Calculate live win probability ball-by-ball")
print("   3. Analyze player performance vs specific teams")
print("   4. Study head-to-head records")
print("   5. Understand venue advantages")
print("\n" + "="*70)

In [ ]:
import torch
print(torch.cuda.is_available())          # Must say: True
print(torch.cuda.get_device_name(0))      # Must say: NVIDIA GeForce RTX 3050 (or similar)
print(torch.cuda.get_device_capability()) # Shows compute capability, e.g., (8, 6) for Ampere